> 在线讲义：[第 17 章 · 长文本外推：PI → NTK-aware → YaRN](https://wikieden.github.io/beyond-transformer/lessons/17-long-context.html) · 本书《超越 Transformer》

# 第 17 章 · 长文本外推：PI → NTK-aware → YaRN

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/17-long-context.html) · 仓库里 `beyond-transformer/notes/lessons/17-long-context.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 没见过的角度 | m·θ_min | `angle_range(...)` | 0.473 → 3.784 |
| 第二步 PI | θ/s | `angle_range(32768, th / 8)` | 回到 0.473 |
| 第三步 NTK-aware | b·s^{d/(d−2)} | `ntk_base(8)` | 82,685；最慢 ÷ 8 |
| 第四步 YaRN | by-parts + 温度 | `by_parts(8)`、`yarn_temp(8)` | 三段；1.2079 |

YaRN 的 NTK-by-parts 按每个平面的波长分别处理，再加注意力温度 √(1/t) = 0.1 ln s + 1。
在小模型上训练、比较 PI / YaRN 推理期缩放的 loss（里程碑 T-M3）留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
D, BASE, L_TRAIN = 128, 10000.0, 4096

**函数 `theta`**：小工具。

In [ ]:
def theta(d=D, base=BASE):
    return base ** (-np.arange(0, d, 2) / d)

**函数 `wavelength`**：小工具。

In [ ]:
def wavelength(d=D, base=BASE):
    return 2 * np.pi / theta(d, base)

**函数 `angle_range`**：位置从 0 到 m_max，最慢那个平面转过的角度（弧度）。

In [ ]:
def angle_range(m_max, th):
    """位置从 0 到 m_max，最慢那个平面转过的角度（弧度）。"""
    return m_max * th[-1]

**函数 `ntk_base`**：NTK-aware：b′ = b · s^{d/(d−2)}，让最慢的平面也正好压慢 s 倍，最快的几乎不动。

In [ ]:
def ntk_base(s, d=D, base=BASE):
    """NTK-aware：b′ = b · s^{d/(d−2)}，让最慢的平面也正好压慢 s 倍，最快的几乎不动。"""
    return base * s ** (d / (d - 2))

**函数 `by_parts`**：YaRN 式 (17)–(18)：r = L / λ（训练长度里转了几圈）。r < α：完全插值（θ/s）；r > β：不动；中间线性过渡。

In [ ]:
def by_parts(s, d=D, base=BASE, L=L_TRAIN, alpha=1.0, beta=32.0):
    """YaRN 式 (17)–(18)：r = L / λ（训练长度里转了几圈）。r < α：完全插值（θ/s）；r > β：不动；中间线性过渡。"""
    th = theta(d, base)
    r = L / wavelength(d, base)
    gamma = np.clip((r - alpha) / (beta - alpha), 0, 1)
    return (1 - gamma) * th / s + gamma * th, r, gamma

**函数 `yarn_temp`**：YaRN 式 (15)：√(1/t) = 0.1 ln s + 1。

In [ ]:
def yarn_temp(s):
    """YaRN 式 (15)：√(1/t) = 0.1 ln s + 1。"""
    return 0.1 * math.log(s) + 1

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
s = 8
th = theta()
print(f"第一步 问题：训练 {L_TRAIN}，要读 {s * L_TRAIN}（s = {s}）")
print(f"  最慢平面在训练长度内转了 {angle_range(L_TRAIN, th):.3f} 弧度；读到 {s * L_TRAIN} 时转到 {angle_range(s * L_TRAIN, th):.3f}（没见过的角度）")
print("\n第二步 PI：位置除以 s")
print(f"  最慢平面：{angle_range(s * L_TRAIN, th / s):.3f} 弧度（回到见过的范围）；最快平面相邻两个位置的角度差 {th[0]:.3f} → {th[0] / s:.3f}（分辨率掉了 s 倍）")

In [ ]:
print("\n第三步 NTK-aware：改底数")
b2 = ntk_base(s)
th2 = theta(base=b2)
print(f"  底数 10000 → {b2:,.0f}；最快平面 θ 不变（{th2[0]:.3f}），最慢平面 {th[-1]:.2e} → {th2[-1]:.2e}（÷ {th[-1] / th2[-1]:.2f}）")

In [ ]:
print("\n第四步 YaRN：按波长分段 + 温度")
th3, r, g = by_parts(s)
for i in (0, 16, 24, 32, 40, 48, 63):
    kind = "不动" if g[i] == 1 else ("完全插值" if g[i] == 0 else "过渡")
    print(f"  第 {i:2d} 个平面：波长 {wavelength()[i]:9.1f}，训练长度里转 r = {r[i]:8.2f} 圈 → {kind}（θ 乘 {th3[i] / th[i]:.3f}）")
print(f"  注意力温度：s = 8 时 √(1/t) = {yarn_temp(8):.4f}，s = 32 时 {yarn_temp(32):.4f}")

## 3. 代码里才看得到的

- **只改 θ，不碰模型**：三种方法都只改 RoPE 的频率表（和一个温度），权重一点不动；所以能直接用在推理上，微调只是让效果更好。
- **这里只算频率**：在模型上测不同长度的 loss、看谁先崩，是里程碑 T-M3。

## 4. 自己改一改

- 把 `by_parts` 的 β 从 32 改成 8，“不动”的平面多了还是少了？
- 把训练长度 L 改成 32768（LLaMA 3 续训后的长度），s = 4 时哪些平面还需要插值？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 只见过半圈的慢钟，读长了转到没见过的角度。
> PI 位置除以 s，慢的回来、快的也糊了。
> NTK 改底数，快的不动、慢的压慢。
> YaRN 看转了几圈：转满三十二圈不动，不到一圈全插值，中间线性过渡；再乘零点一 ln s 加一的温度。